In [8]:
from typing import Literal, Optional, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, StateGraph
from langgraph.types import Command, interrupt




In [9]:
class ApprovalState(TypedDict):
    action_details: str
    status: Optional[Literal["pending", "approved", "rejected"]]


In [3]:
def proceed_node(state: ApprovalState):
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    return {"status": "rejected"}

In [7]:
def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    decission = interrupt(
        {
            "question": "Approve this action?",
            "details": state["action_details"],
        }
    )

    return Command(goto="proceed" if decission else "cancel")

In [10]:
builder = StateGraph(ApprovalState)
builder.add_node("approval",approval_node)
builder.add_node("proceed",proceed_node)
builder.add_node("cancel",cancel_node)

builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)


In [11]:
config = {"configurable": {"thread_id": "approval-123"}}
initial = graph.invoke(
    {"action_details": "Transfer $500", "status": "pending"},
    config=config,
)

In [13]:
print(initial["__interrupt__"])

[Interrupt(value={'question': 'Approve this action?', 'details': 'Transfer $500'}, id='3f76f4dc67c89b8e6b21da8ebf1b6953')]


In [14]:
resumed = graph.invoke(Command(resume=True), config=config)
print(resumed["status"])  # -> "approved"


approved


In [15]:
print(resumed)

{'action_details': 'Transfer $500', 'status': 'approved'}


In [18]:
import sqlite3
from typing import TypedDict

from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
